# fine-tuning 

- 일부 또는 전체 가중치를 조정하여 새로운 데이터셋에 맞게 최적화 하는 방식
- 기존 학습된 모델의 강력한 특징 추출 능력을 활용하는 특징 추출과 달리, 파인 튜닝은 모델의 가중치를 업데이트하여 새로운 데이터에 적합한 더 정교한 모델을 만들 수 있다. 


- 전체 모델을 미세 조정할 수도 있고, 출력층에 가까운 몇개의 레이어만 조정할 수 있다. 
- Pooling 층: 미세조정에서는 이부분의 가중치도 업데이트 
- 완전 연결층(convolution layer + ReLU ) : 가중치를 처음부터 다시 학습하거나 기존 가중치를 미세 조정 
- 학습 과정 : 모든 레이어를 업데이트 하는 것이 일반적

## 특징 추출과의 차이점 

- 가중치 업데이트 범위
    - 특징 추출은 사전 학습된 모델의 대부분의 가중치를 고정하고, 출력 레이어만 학습 
- 반면 파인튜닝은 모델 전체 또는 일부 레이어의 가중치를 새 데이터에 맞게 업데이트 
    - 더 정교한 모델 조정이 가능, 하지만 더 많은 가중치를 업데이트 함에 따라 학습시간이 길어지고, 오버피팅의 위험이 높아짐. 

## 파인튜닝이 필요한 상황

- 사전 학습 된 모델과 새로운 데이터셋이 매우 다를때 
- 더 높은 성능이 요구 
- 데이터셋 크기가 충분할 때 
    - 파인튜닝은 더 많은 파라미터를 학습하기 때문에 충분한 양의 데이터가 있어야 효과적 

## 장점
- 더 높은 성능 
- 적응력 
- 세밀한 조정 가능 ( 고차원적인 특징에만 영향을 줄 수 있어, 효율적인 학습이 가능 )

## 위험요소 

- 오버피팅 가능성
- 일반화 성능이 떨어질 위험 
- 긴 학습 시간 ( 컴퓨팅 자원이 많이 필요 )
- 하이퍼파라미터 튜닝의 어려움 
    - 파인 튜닝 과정에서 학습률(lr), 조정할 레이어의 범위 등 여러 하이퍼파라미터를 적절히 설정해야 하며, 잘못 설정할 경우 학습이 제대로 이루어지지 않거나 성능이 저하 될 수 있다. 

In [2]:
import torch
import torchvision.models as models

model = models.resnet18(pretrained=True)
print(model)

/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/donghun2/workspace/machine_learning/dacon/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [3]:
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: True
bn1.weight: True
bn1.bias: True
layer1.0.conv1.weight: True
layer1.0.bn1.weight: True
layer1.0.bn1.bias: True
layer1.0.conv2.weight: True
layer1.0.bn2.weight: True
layer1.0.bn2.bias: True
layer1.1.conv1.weight: True
layer1.1.bn1.weight: True
layer1.1.bn1.bias: True
layer1.1.conv2.weight: True
layer1.1.bn2.weight: True
layer1.1.bn2.bias: True
layer2.0.conv1.weight: True
layer2.0.bn1.weight: True
layer2.0.bn1.bias: True
layer2.0.conv2.weight: True
layer2.0.bn2.weight: True
layer2.0.bn2.bias: True
layer2.0.downsample.0.weight: True
layer2.0.downsample.1.weight: True
layer2.0.downsample.1.bias: True
layer2.1.conv1.weight: True
layer2.1.bn1.weight: True
layer2.1.bn1.bias: True
layer2.1.conv2.weight: True
layer2.1.bn2.weight: True
layer2.1.bn2.bias: True
layer3.0.conv1.weight: True
layer3.0.bn1.weight: True
layer3.0.bn1.bias: True
layer3.0.conv2.weight: True
layer3.0.bn2.weight: True
layer3.0.bn2.bias: True
layer3.0.downsample.0.weight: True
layer3.0.downsample.1.weight: T

In [4]:
#마지막 레이어 들만 학습 가능하게 설정 
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.downsample.0.weight: 

In [5]:
#일부 마지막 파라미터들만 학습 가능하게 설정 
for name, param in model.named_parameters():
    param.requires_grad = False

In [7]:
#마지막 10개의 파라미터만 학습이 가능하도록
#역전파 과정에서 가중치가 업데이트 됨. 새로운 데이터에 맞춰 학습 
for name, param in list(model.named_parameters())[-10:]:
    param.requires_grad = True
    
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.downsample.0.weight: 

## 스케쥴 파인튜닝 

- 에포크에서 추가적인 레이어들을 학습 가능하게 설정하는 방법 
- 초기에는 마지막 레이어만 학습 가능하게 하고, 에포크가 특정 값 이상일 때 추가 레이어를 학습 가능하게 하는 것 


In [8]:
for name, param in model.named_parameters():
    param.requires_grad = False
epoch = 1

# 초기에는 마지막 레이어들만 학습 가능하게 설정
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# 일정 에포크 이후, 추가적인 레이어들을 학습 가능하게 설정
if epoch > 2:
    for name, param in model.named_parameters():
        if "layer3" in name:  
            param.requires_grad = True
            
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.downsample.0.weight: 

In [9]:
epoch = 3

for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

if epoch > 2:
    for name, param in model.named_parameters():
        if "layer3" in name:  
            param.requires_grad = True
            
for name, param in model.named_parameters():
    print(f'{name}: {param.requires_grad}')

conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: True
layer3.0.bn1.weight: True
layer3.0.bn1.bias: True
layer3.0.conv2.weight: True
layer3.0.bn2.weight: True
layer3.0.bn2.bias: True
layer3.0.downsample.0.weight: True
l

In [10]:
for param in model.parameters():
    param.requires_grad = False

layer_names = ["fc", "layer4", "layer3", "layer2"]
trainable_layers = [model.fc, model.layer4, model.layer3, model.layer2]

for i, layer in enumerate(trainable_layers):
    print(f"\n단계 {i+1}: {layer_names[i]} 레이어 해제")
    
    for param in layer.parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        print(f'{name}: {param.requires_grad}')
        
print("\n점진적 언프리징 완료")


단계 1: fc 레이어 해제
conv1.weight: False
bn1.weight: False
bn1.bias: False
layer1.0.conv1.weight: False
layer1.0.bn1.weight: False
layer1.0.bn1.bias: False
layer1.0.conv2.weight: False
layer1.0.bn2.weight: False
layer1.0.bn2.bias: False
layer1.1.conv1.weight: False
layer1.1.bn1.weight: False
layer1.1.bn1.bias: False
layer1.1.conv2.weight: False
layer1.1.bn2.weight: False
layer1.1.bn2.bias: False
layer2.0.conv1.weight: False
layer2.0.bn1.weight: False
layer2.0.bn1.bias: False
layer2.0.conv2.weight: False
layer2.0.bn2.weight: False
layer2.0.bn2.bias: False
layer2.0.downsample.0.weight: False
layer2.0.downsample.1.weight: False
layer2.0.downsample.1.bias: False
layer2.1.conv1.weight: False
layer2.1.bn1.weight: False
layer2.1.bn1.bias: False
layer2.1.conv2.weight: False
layer2.1.bn2.weight: False
layer2.1.bn2.bias: False
layer3.0.conv1.weight: False
layer3.0.bn1.weight: False
layer3.0.bn1.bias: False
layer3.0.conv2.weight: False
layer3.0.bn2.weight: False
layer3.0.bn2.bias: False
layer3.0.down